"""
fact_po_descriptions — Generator
=================================
Adds a rich natural-language description to each purchase order.
 
Design goals:
  • 8 semantically distinct clusters, grounded in real PO attributes
  • 60-130 words per description (within BERT's 512-token window)
  • Vocabulary variation within clusters so clustering is non-trivial
  • Cluster assignment driven by PO signals (urgency, seasonality, fraud,
    sanctions, transport mode, etc.) — not random
 
Cluster taxonomy
----------------
  1  Urgent / Emergency Procurement
  2  Routine Replenishment
  3  Strategic Sourcing & Supplier Development
  4  Quality Issue Resolution & Returns
  5  Sanctions & Geopolitical Risk Mitigation
  6  Seasonal Demand Buildup
  7  Bulk Cost Optimisation
  8  Regulatory & Customs Compliance
"""

In [2]:
import numpy as np
import pandas as pd
import random
import os
from datetime import date, timedelta

In [3]:
rng = np.random.default_rng(seed=99)
random.seed(99)
 
OUTPUT_DIR = r"C:\Courses\Innopolis\2026\ВЭД\Данные2"
START_DATE = date(2019, 1, 1)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. Load dimension tables for contextual text generation
# ─────────────────────────────────────────────────────────────
df_sup   = pd.read_csv(f"{OUTPUT_DIR}/dim_suppliers.csv").drop_duplicates()
df_prod  = pd.read_csv(f"{OUTPUT_DIR}/dim_products.csv")
df_sp    = pd.read_csv(f"{OUTPUT_DIR}/dim_supplier_products.csv")
df_modes = pd.read_csv(f"{OUTPUT_DIR}/dim_transport_modes.csv")
df_cntry = pd.read_csv(f"{OUTPUT_DIR}/dim_countries.csv")
 
# Lookup helpers
sup_map   = df_sup.set_index("supplier_id").to_dict("index")
prod_map  = df_prod.set_index("product_id").to_dict("index")
mode_map  = df_modes.set_index("transport_mode_id").to_dict("index")
cntry_map = df_cntry.set_index("country_id").to_dict("index")

In [31]:
df_po = pd.read_csv(f"{OUTPUT_DIR}\\fact_purchase_orders_clean.csv").sort_values(by = "po_id")

df_suppliers = (
    pd.read_csv(f"{OUTPUT_DIR}\\dim_suppliers.csv")
    .drop_duplicates()
    .loc[:, ["supplier_id", "country_id", "years_of_relationship", "maturity_score"]]
)

df_countries = (
    pd.read_csv(f"{OUTPUT_DIR}\\dim_countries.csv")
    .loc[:, ["country_id", "country_risk_base", "sanctions_risk", "port_efficiency_base", "avg_customs_days"]]
)

df_products = (
    pd.read_csv(f"{OUTPUT_DIR}\\dim_products.csv")
    .loc[:, ["product_id", "category_id"]]
)

df_transport_modes = (
    pd.read_csv(f"{OUTPUT_DIR}\\dim_transport_modes.csv")
    .loc[:, ["transport_mode_id", "avg_transit_days", "delay_probability"]]
)

df_po = (
    df_po
    .merge(df_suppliers, how = "left", on = "supplier_id")
    .merge(df_countries, how = "left", on = "country_id")
    .merge(df_products, how = "left", on = "product_id")
    .merge(df_transport_modes, how = "left", on = "transport_mode_id")
)

In [35]:
# ─────────────────────────────────────────────────────────────
# 3. Cluster assignment — grounded in PO signals
# ─────────────────────────────────────────────────────────────
 
val_q75 = df_po["total_value_usd"].quantile(0.75)
val_q25 = df_po["total_value_usd"].quantile(0.25)
 
HIGH_RISK_COUNTRIES = {"BY", "RU", "US"}   # sanctions-sensitive post-2022
NEAR_SHORE          = {"DE", "PL", "BY", "KZ", "TR"}
 
def assign_cluster(row):
    """
    Returns (cluster_id, cluster_name) based on PO attributes.
    Weights are stochastic so clusters overlap naturally, as in real data.
    """
    weights = [1.0] * 8   # base weight for each cluster (1-indexed → idx 0-7)
 
    month = pd.Timestamp(row["order_date"]).month
    shock = row["shock_index_at_order"]
    rel   = row["supplier_reliability"]
    mode  = row["transport_mode_id"]
    fraud = row["is_fraud"]
    ft    = row["fraud_type"]
    val   = row["total_value_usd"]
    yrs   = row["years_of_relationship"]
    late  = (row["actual_delivery_days"] or 0) > row["expected_delivery_days"] * 1.2
    cntry = row["country_id"]
    exp   = row["expected_delivery_days"]
 
    # Cluster 1 — Urgent
    if mode == 3:          weights[0] += 8   # air freight ≈ urgency
    if shock > 0.5:        weights[0] += 4
    if late:               weights[0] += 3
    if rel < 0.5:          weights[0] += 2
 
    # Cluster 2 — Routine
    if 0.6 <= rel <= 0.9:  weights[1] += 5
    if yrs > 3:            weights[1] += 4
    if mode in (1, 4):     weights[1] += 2   # sea FCL / rail → scheduled
    if shock < 0.2:        weights[1] += 3
 
    # Cluster 3 — Strategic Sourcing
    if yrs < 1.5:          weights[2] += 7   # new supplier
    if val < val_q25:      weights[2] += 3   # small pilot order
    if not fraud:          weights[2] += 2
 
    # Cluster 4 — Quality Issue
    if fraud and ft in ("underinvoice", "phantom"):
        weights[3] += 9
    if rel < 0.55:         weights[3] += 5
    if late:               weights[3] += 3
 
    # Cluster 5 — Sanctions / Geopolitical
    if cntry in HIGH_RISK_COUNTRIES: weights[4] += 8
    if shock > 0.6:        weights[4] += 6
    if fraud and ft == "overinvoice": weights[4] += 3
    if pd.Timestamp(row["order_date"]).year >= 2022: weights[4] += 2
 
    # Cluster 6 — Seasonal Demand
    if month in (9, 10, 11, 12): weights[5] += 7   # Q4 buildup
    if month in (1, 2):          weights[5] += 3   # post-CNY restocking
    if val > val_q75:            weights[5] += 2
 
    # Cluster 7 — Bulk / Cost Optimisation
    if val > val_q75:      weights[6] += 7
    if mode == 1:          weights[6] += 3   # FCL = full container = bulk
    if yrs > 4:            weights[6] += 2
    if not fraud:          weights[6] += 1
 
    # Cluster 8 — Regulatory / Compliance
    if exp > 30:           weights[7] += 5   # long customs = complex compliance
    if cntry not in NEAR_SHORE: weights[7] += 3
    if shock > 0.3:        weights[7] += 2
    if pd.Timestamp(row["order_date"]).year in (2022, 2023): weights[7] += 2
 
    w = np.array(weights, dtype=float)
    w /= w.sum()
    idx = int(rng.choice(8, p=w))
    return idx + 1   # 1-indexed
 
CLUSTER_NAMES = {
    1: "Urgent / Emergency Procurement",
    2: "Routine Replenishment",
    3: "Strategic Sourcing & Supplier Development",
    4: "Quality Issue Resolution",
    5: "Sanctions & Geopolitical Risk Mitigation",
    6: "Seasonal Demand Buildup",
    7: "Bulk Cost Optimisation",
    8: "Regulatory & Customs Compliance",
}
 
df_po["true_cluster_id"]   = df_po.apply(assign_cluster, axis=1)
df_po["true_cluster_name"] = df_po["true_cluster_id"].map(CLUSTER_NAMES)
print("Cluster distribution:")
print(df_po["true_cluster_name"].value_counts().to_string())

Cluster distribution:
true_cluster_name
Routine Replenishment                        745
Bulk Cost Optimisation                       413
Regulatory & Customs Compliance              399
Seasonal Demand Buildup                      311
Strategic Sourcing & Supplier Development    303
Urgent / Emergency Procurement               254
Quality Issue Resolution                     195
Sanctions & Geopolitical Risk Mitigation     176


In [36]:
# ─────────────────────────────────────────────────────────────
# 4. Vocabulary pools — rich, varied, domain-accurate
# ─────────────────────────────────────────────────────────────
 
URGENCY_OPENERS = [
    "An urgent procurement request has been raised following an unexpected stockout of {product} in our Moscow warehouse.",
    "Emergency sourcing action initiated after {product} inventory fell below the critical safety threshold ahead of a key client delivery.",
    "A rush order for {product} has been placed with {supplier} to prevent an imminent production line stoppage.",
    "Expedited procurement triggered by an unplanned spike in customer demand for {product} that depleted buffer stock ahead of schedule.",
    "Critical shortage of {product} identified during the weekly inventory audit; emergency purchase order issued immediately.",
    "Our internal logistics team escalated a stockout alert for {product}, prompting an out-of-cycle purchase with {supplier}.",
]
URGENCY_MIDDLES = [
    "Air freight was selected over standard sea routing to reduce transit time from {country} despite the significant cost premium.",
    "The procurement team invoked the emergency supplier protocol, bypassing the standard three-quote requirement.",
    "{supplier} confirmed same-week dispatch capacity and prioritised our order over other clients.",
    "Given the time-critical nature of this purchase, payment terms were agreed on a prepayment basis to accelerate processing.",
    "Internal approval was fast-tracked through the emergency escalation path, receiving CFO sign-off within 24 hours.",
    "The supply chain director authorised a 35% cost premium over the standard unit price to secure immediate availability.",
]
URGENCY_CLOSERS = [
    "All customs pre-clearance documents were submitted in advance to minimise border delays upon arrival.",
    "The warehouse has been placed on standby for same-day receiving and put-away upon delivery.",
    "A post-incident review is scheduled to assess safety stock levels and prevent recurrence.",
    "An interim safety stock increase of 20% has been proposed to the procurement committee to prevent similar shortages.",
    "This order will be tracked daily and escalated to the COO if transit milestones are missed.",
    "Contract clause for expedited delivery has been activated; supplier penalty applies if delivery date is not met.",
]
 
ROUTINE_OPENERS = [
    "This is a scheduled replenishment order for {product} placed as part of the quarterly procurement cycle with {supplier}.",
    "Standard monthly reorder of {product} in accordance with the approved annual supply agreement with {supplier}.",
    "Routine restocking of {product} triggered by the ERP system upon reaching the reorder point.",
    "Periodic replenishment order for {product} issued under the long-term framework contract with {supplier}.",
    "This purchase order represents the regular bi-monthly supply of {product} from our incumbent supplier {supplier}.",
    "Scheduled procurement of {product} as per the rolling 13-week demand forecast reviewed last week.",
]
ROUTINE_MIDDLES = [
    "Delivery is routed via {mode} as per the standard logistics arrangement in place since {year}.",
    "Unit pricing has been confirmed at the contracted rate with no variance from the last three orders.",
    "The order quantity of {qty} units was determined by the standard min-max replenishment formula.",
    "No deviations from the standard purchase conditions are noted; all terms remain unchanged.",
    "{supplier} has a consistent on-time delivery record for this product line over the past {years} years.",
    "Order placed 21 days in advance of projected stockout to maintain the standard safety stock buffer.",
]
ROUTINE_CLOSERS = [
    "This order requires no additional approvals; it falls within the category manager's delegated authority.",
    "Payment will be processed on standard {pt}-day terms upon receipt of goods and invoice.",
    "A copy of the quality certificate is required to accompany the shipment as per standard protocol.",
    "The procurement analyst confirmed that no alternative sourcing actions are required at this time.",
    "Order logged in the ERP system and forwarded to the logistics team for inbound planning.",
    "Supplier performance will be reviewed at the next quarterly business review in line with KPIs.",
]
 
STRATEGIC_OPENERS = [
    "This order is part of a strategic initiative to qualify {supplier} in {country} as a secondary source for {product}.",
    "Pilot procurement placed with {supplier} to evaluate their capability to supply {product} at commercial scale.",
    "First purchase order issued to {supplier} as part of the supplier diversification programme for the {category} category.",
    "The procurement strategy team identified {supplier} as a potential long-term partner for {product} and authorised this evaluation order.",
    "New supplier onboarding order for {product} following a successful pre-qualification audit of {supplier}'s facility in {city}.",
    "This order has been raised to stress-test {supplier}'s fulfilment process before awarding a strategic framework agreement.",
]
STRATEGIC_MIDDLES = [
    "The quantity ordered is intentionally modest to minimise risk exposure during the qualification phase.",
    "Quality inspection at origin has been arranged with a third-party inspector prior to shipment.",
    "The supplier was identified through a market benchmark exercise that assessed 12 potential vendors across {region}.",
    "A cross-functional evaluation team including procurement, quality, and logistics has been assigned to this supplier trial.",
    "The supplier has submitted all required onboarding documentation including financial statements and compliance certificates.",
    "Concurrently, our incumbent supplier continues to fulfil demand at full volume while this trial is conducted.",
]
STRATEGIC_CLOSERS = [
    "Results from this order will inform the make-or-buy decision for {product} in the next budget cycle.",
    "A formal supplier scorecard will be issued after delivery, covering quality, lead time, communication, and documentation.",
    "If performance meets the defined threshold, a 12-month framework agreement will be proposed at the next review.",
    "The procurement director will sign off the final supplier approval based on the post-order evaluation report.",
    "This engagement aligns with the company's goal of reducing single-source dependency in critical product categories.",
    "The supplier has been informed that this order carries strategic significance and that performance will be closely monitored.",
]
 
QUALITY_OPENERS = [
    "This replacement order for {product} has been raised following the rejection of the previous shipment from {supplier} due to quality non-conformance.",
    "A quality hold was placed on the last batch of {product} received from {supplier}; this order covers the shortfall while the root cause is investigated.",
    "Purchase order issued to address a confirmed defect in {product} supplied by {supplier}, affecting {qty} units from the previous delivery.",
    "The quality control team issued a non-conformance report for {product} from {supplier}; this order restores inventory to operational levels.",
    "Following a customer complaint regarding {product} sourced from {supplier}, a replacement procurement has been authorised.",
    "Internal audit identified a certification mismatch for {product} supplied by {supplier}; this corrective order ensures compliant stock availability.",
]
QUALITY_MIDDLES = [
    "The defective units have been quarantined and a formal 8D corrective action report has been requested from {supplier}.",
    "The supplier has been placed on a provisional watch list pending the outcome of the root cause analysis.",
    "An on-site audit of {supplier}'s production line in {city} is being arranged by the quality assurance team.",
    "The unit price on this replacement order reflects a 10% credit agreed with {supplier} as partial compensation for disruption.",
    "Third-party testing of the rejected batch has been commissioned to provide independent verification of the defect claim.",
    "This is the second quality incident with {supplier} within 12 months, triggering a mandatory supplier improvement plan.",
]
QUALITY_CLOSERS = [
    "Future shipments from {supplier} will require a pre-shipment inspection certificate until further notice.",
    "The procurement team will review the continuation of this supplier relationship at the next category strategy meeting.",
    "A debit note for the cost of the quality failure, including rework and logistics, has been submitted to {supplier}.",
    "Approval for this replacement order was granted under the quality emergency protocol by the Head of Procurement.",
    "Supplier has acknowledged the non-conformance and committed to a corrective action plan within 30 days.",
    "All received units from this replacement order will undergo 100% incoming inspection upon arrival.",
]
 
SANCTIONS_OPENERS = [
    "This purchase order has been re-routed through {supplier} in {country} following the imposition of new trade restrictions affecting our previous source.",
    "Geopolitical developments in the supply region have necessitated an urgent shift in sourcing strategy for {product}, resulting in this order with {supplier}.",
    "Following sanctions-related disruptions to our supply chain, this order represents a transition to a compliant alternative supplier for {product}.",
    "The compliance team has confirmed that {supplier} in {country} meets all current export control requirements; this order proceeds under the approved alternative sourcing plan.",
    "In response to the evolving regulatory environment, procurement has engaged {supplier} as a sanctions-compliant source for {product} on an interim basis.",
    "Trade compliance screening flagged elevated risk for the original supply route; this order with {supplier} was approved as a geopolitically neutral alternative.",
]
SANCTIONS_MIDDLES = [
    "All required end-user certificates and export licence declarations have been completed and filed with this order.",
    "The trade compliance officer conducted a dual-use goods screening; no restrictions apply to {product} under current HS code {hs}.",
    "Payment routing has been reviewed and confirmed clear of OFAC, EU, and UK sanctions lists.",
    "The logistics route via {country} was selected to avoid transit through jurisdictions currently under comprehensive sanctions.",
    "A legal opinion on the permissibility of this transaction was obtained before the order was raised.",
    "An enhanced due diligence file has been opened for {supplier} in accordance with the company's sanctions risk policy.",
]
SANCTIONS_CLOSERS = [
    "This order will be reviewed at 90 days to assess whether the geopolitical situation has stabilised sufficiently to return to the original supply chain.",
    "All shipping documents must explicitly state the country of origin; co-mingling with restricted-origin goods is prohibited.",
    "The CFO and General Counsel have co-signed the procurement authorisation given the elevated compliance sensitivity of this transaction.",
    "A sanctions compliance attestation has been obtained from {supplier} and is held on file.",
    "Procurement will brief the executive committee on the broader supply chain restructuring implications at the next monthly review.",
    "Ongoing monitoring of the geopolitical situation has been assigned to the trade compliance team with weekly reporting.",
]
 
SEASONAL_OPENERS = [
    "This order is part of the annual Q4 inventory build programme for {product}, timed to meet the anticipated year-end demand surge.",
    "Pre-season procurement of {product} placed with {supplier} to ensure sufficient stock ahead of the peak trading period.",
    "Seasonal demand forecasting indicates a 40% uplift in {product} consumption from October to December; this order pre-positions inventory accordingly.",
    "This purchase order covers the post-Chinese New Year restocking of {product}, addressing the supply gap created by {supplier}'s factory closure period.",
    "Advance procurement of {product} ahead of the peak shipping season, when freight capacity from {country} becomes constrained and lead times extend significantly.",
    "The demand planning team has flagged elevated Q4 requirements for {product}; this order ensures we are positioned ahead of the congestion window.",
]
SEASONAL_MIDDLES = [
    "Order placed 6 weeks earlier than standard to account for the seasonal extension in transit times from {country} during peak shipping periods.",
    "The quantity of {qty} units represents a 30% increase over the standard order to cover projected seasonal uplift.",
    "Warehouse capacity has been pre-allocated at our Moscow distribution centre to accommodate this seasonal stock build.",
    "Sea freight capacity was booked 8 weeks in advance to secure favourable rates before the Q4 premium applies.",
    "This order has been coordinated with sales forecasting to align inventory build with confirmed customer pre-orders.",
    "Port congestion historically peaks in Q4 on this route; the logistics team has built in a 10-day buffer to the planned receipt date.",
]
SEASONAL_CLOSERS = [
    "Excess stock following the peak season will be liquidated in the January sale or held at standard inventory carrying cost.",
    "Insurance cover has been extended to reflect the elevated inventory value during the seasonal build period.",
    "The procurement committee approved the seasonal stock build under the annual budget allocation for forward-buying.",
    "A review of sell-through rates will inform the volume decision for the equivalent period next year.",
    "Vendor managed inventory arrangements with {supplier} are under discussion to streamline future seasonal replenishment.",
    "The commercial team has been notified of the expected inbound date to align customer commitment schedules.",
]
 
BULK_OPENERS = [
    "This consolidated bulk order for {product} was negotiated with {supplier} to leverage the volume discount threshold and reduce the per-unit landed cost.",
    "Following a spend analysis across multiple business units, procurement has aggregated demand for {product} into a single high-volume order with {supplier}.",
    "An opportunistic bulk purchase of {product} has been authorised to take advantage of a favourable pricing window offered by {supplier}.",
    "This order consolidates six separate internal requisitions for {product} into one shipment to optimise freight costs and supplier pricing.",
    "The category manager negotiated a {pct}% volume rebate with {supplier} contingent on a minimum order of {qty} units, which has been achieved through demand pooling.",
    "Strategic inventory investment authorised for {product} based on a favourable forward price from {supplier} and strong demand visibility for the next two quarters.",
]
BULK_MIDDLES = [
    "A full container load has been arranged to minimise the per-unit freight cost on this high-volume shipment.",
    "The total order value of {val} USD exceeds the standard approval threshold and has been reviewed by the procurement director.",
    "Payment terms of {pt} days have been negotiated as part of the volume commitment, improving working capital by an estimated {wc} days.",
    "Inventory carrying cost analysis confirmed that the bulk discount outweighs the additional warehousing costs over the holding period.",
    "The finance team has conducted a net present value assessment and confirmed a positive return on the forward-buy investment.",
    "This consolidation eliminates three separate logistics movements, reducing the carbon footprint of the category by an estimated 18%.",
]
BULK_CLOSERS = [
    "Storage at our third-party logistics provider has been pre-booked to accommodate the elevated inbound volume.",
    "The stock will be released to the production floor on a first-in-first-out basis aligned with the monthly consumption plan.",
    "A post-delivery variance analysis will be conducted to validate the actual savings against the procurement business case.",
    "The supplier has been requested to stagger delivery in two tranches to ease inbound handling capacity.",
    "This bulk purchase has been flagged to the treasury team for working capital planning given the above-average order value.",
    "The commercial director has signed off the exceptional purchase authority form required for orders above the standard category limit.",
]
 
COMPLIANCE_OPENERS = [
    "This purchase order has been structured to comply with updated import regulations affecting {product} sourced from {country}, effective this quarter.",
    "Procurement has raised this order following confirmation from the customs broker that the revised HS classification for {product} now requires additional documentation.",
    "In response to new phytosanitary and technical standards applicable to {product} imports from {country}, this order includes enhanced compliance provisions.",
    "This order reflects the updated sourcing requirements following a regulatory review of {product}'s compliance status under current import licensing rules.",
    "The legal and compliance team confirmed that {product} imported from {country} is now subject to mandatory pre-arrival notification; this order incorporates those requirements.",
    "Following a customs audit at our facility, procurement has restructured the terms and documentation for {product} orders from {supplier} to ensure full regulatory compliance.",
]
COMPLIANCE_MIDDLES = [
    "All required import licences, certificates of conformity, and technical data sheets have been obtained and attached to this order.",
    "The goods will undergo mandatory customs inspection at the port of entry; additional transit time has been incorporated into the planned delivery schedule.",
    "A customs broker has been engaged specifically for this shipment to navigate the additional procedural requirements at the Russian border.",
    "The HS code for {product} has been formally reclassified in our ERP system to reflect the latest binding tariff information ruling.",
    "Anti-dumping duty implications were reviewed by the trade compliance team; no additional duties apply under current regulations.",
    "The supplier has been instructed to prepare all export documentation in strict accordance with the updated import requirements.",
]
COMPLIANCE_CLOSERS = [
    "Copies of all compliance certificates will be filed in the regulatory document management system upon receipt of goods.",
    "A post-clearance audit is expected from the customs authority within 60 days; all supporting documents are being retained accordingly.",
    "The procurement legal team will monitor further regulatory changes in {country} that may affect future orders in this category.",
    "This order has been reviewed by both the legal and finance teams; no additional VAT or excise exposure has been identified.",
    "The regulatory compliance cost of {val_share}% has been incorporated into the total landed cost model for this product.",
    "Training on the updated import requirements has been provided to the logistics team to ensure consistent handling of future shipments.",
]
 
CLUSTER_POOLS = {
    1: (URGENCY_OPENERS,   URGENCY_MIDDLES,   URGENCY_CLOSERS),
    2: (ROUTINE_OPENERS,   ROUTINE_MIDDLES,   ROUTINE_CLOSERS),
    3: (STRATEGIC_OPENERS, STRATEGIC_MIDDLES, STRATEGIC_CLOSERS),
    4: (QUALITY_OPENERS,   QUALITY_MIDDLES,   QUALITY_CLOSERS),
    5: (SANCTIONS_OPENERS, SANCTIONS_MIDDLES, SANCTIONS_CLOSERS),
    6: (SEASONAL_OPENERS,  SEASONAL_MIDDLES,  SEASONAL_CLOSERS),
    7: (BULK_OPENERS,      BULK_MIDDLES,      BULK_CLOSERS),
    8: (COMPLIANCE_OPENERS,COMPLIANCE_MIDDLES,COMPLIANCE_CLOSERS),
}
 
# ─────────────────────────────────────────────────────────────
# 5. Description synthesis
# ─────────────────────────────────────────────────────────────
 
def pick(pool):
    return random.choice(pool)
 
def fill(template, ctx):
    """Safe format — skips unknown keys."""
    try:
        return template.format(**ctx)
    except KeyError:
        return template
 
def make_description(row):
    cluster  = int(row["true_cluster_id"])
    openers, middles, closers = CLUSTER_POOLS[cluster]
 
    sup      = sup_map.get(int(row["supplier_id"]), {})
    prod     = prod_map.get(int(row["product_id"]), {})
    mode     = mode_map.get(int(row["transport_mode_id"]), {})
    cntry_id = str(row["country_id"])
    cntry    = cntry_map.get(cntry_id, {})
 
    ctx = {
        "product":    prod.get("product_name", "the goods"),
        "supplier":   sup.get("supplier_name",  "the supplier"),
        "country":    cntry.get("country_name", cntry_id),
        "city":       sup.get("city", "the supplier's facility"),
        "region":     cntry.get("region", "the region"),
        "category":   prod.get("category_id", ""),
        "mode":       mode.get("mode_name", "the selected transport mode"),
        "qty":        f"{int(row['quantity']):,}",
        "val":        f"${int(row['total_value_usd']):,}",
        "val_share":  f"{round(rng.uniform(1.5, 4.5), 1)}",
        "hs":         prod.get("hs_code", ""),
        "year":       str(pd.Timestamp(row["order_date"]).year - int(rng.integers(1, 4))),
        "years":      str(int(row["years_of_relationship"])),
        "pt":         str(random.choice([30, 45, 60, 90])),
        "pct":        str(int(rng.integers(5, 20))),
        "wc":         str(int(rng.integers(10, 30))),
    }
 
    # Draw 1 opener + 1-2 middles + 1 closer
    n_mid = int(rng.integers(1, 3))
    sentences = (
        [fill(pick(openers), ctx)]
        + [fill(pick(middles), ctx) for _ in range(n_mid)]
        + [fill(pick(closers), ctx)]
    )
 
    return " ".join(sentences)
 
# ─────────────────────────────────────────────────────────────
# 6. Build the table
# ─────────────────────────────────────────────────────────────
 
desc_rows = []
for _, row in df_po.iterrows():
    text = make_description(row)
    desc_rows.append({
        "description_id":   int(row["po_id"]),
        "po_id":            int(row["po_id"]),
        "order_date":       row["order_date"],
        "description_text": text,
        "word_count":       len(text.split()),
        "char_count":       len(text),
        "true_cluster_id":  int(row["true_cluster_id"]),
        "true_cluster_name": row["true_cluster_name"],
    })
 
df_desc = pd.DataFrame(desc_rows)
 
# ─────────────────────────────────────────────────────────────
# 7. Inject mild noise (realistic data quality)
# ─────────────────────────────────────────────────────────────
 
# ~2% of texts get a truncation (simulates DB varchar overflow)
trunc_idx = rng.choice(len(df_desc), size=int(len(df_desc)*0.02), replace=False)
for i in trunc_idx:
    df_desc.loc[i, "description_text"] = df_desc.loc[i, "description_text"][:120] + "..."
 
# ~1% are near-duplicates (copy-paste reuse from a previous order)
dup_idx = rng.choice(len(df_desc), size=int(len(df_desc)*0.01), replace=False)
if len(dup_idx) > 0:
    src = df_desc.loc[0, "description_text"]
    for i in dup_idx:
        df_desc.loc[i, "description_text"] = src
 
# ~0.5% have empty descriptions (missing data)
null_idx = rng.choice(len(df_desc), size=int(len(df_desc)*0.005), replace=False)
for i in null_idx:
    df_desc.loc[i, "description_text"] = ""
 
# Recalculate word/char counts after noise
df_desc["word_count"] = df_desc["description_text"].str.split().str.len().fillna(0).astype(int)
df_desc["char_count"] = df_desc["description_text"].str.len().fillna(0).astype(int)

In [43]:
df_desc[["po_id", "description_text"]].to_csv(f"{OUTPUT_DIR}\\fact_purchase_order_descriptions.csv", index = False)

In [ ]:

# ─────────────────────────────────────────────────────────────
# 9. Minimal BERT usage guide saved alongside data
# ─────────────────────────────────────────────────────────────
 
GUIDE = '''# BERT Semantic Clustering — Usage Guide
## Table: fact_po_descriptions
 
### Schema
| Column | Type | Description |
|--------|------|-------------|
| description_id | INT | PK |
| po_id | INT | FK → fact_purchase_orders |
| order_date | DATE | Order date |
| description_text | TEXT | Natural language PO description (~80 words) |
| word_count | INT | Word count (0 = missing data) |
| char_count | INT | Character count |
| true_cluster_id | INT | Ground truth cluster label (1–8) |
| true_cluster_name | VARCHAR | Human-readable cluster name |
 
### 8 Semantic Clusters
| ID | Name | Key signals in text |
|----|------|-------------------|
| 1 | Urgent / Emergency | "stockout", "emergency", "expedited", "air freight", "critical" |
| 2 | Routine Replenishment | "scheduled", "quarterly", "standard reorder", "framework contract" |
| 3 | Strategic Sourcing | "new supplier", "qualification", "pilot", "diversification" |
| 4 | Quality Issue | "rejection", "non-conformance", "defective", "replacement", "audit" |
| 5 | Sanctions & Geopolitical | "sanctions", "trade compliance", "OFAC", "re-routed", "geopolitical" |
| 6 | Seasonal Demand | "Q4 buildup", "peak season", "Chinese New Year", "seasonal" |
| 7 | Bulk Cost Optimisation | "consolidated", "volume discount", "bulk", "rebate", "FCL" |
| 8 | Regulatory & Compliance | "import licence", "HS code", "customs inspection", "regulatory" |
 
---
 
### Recommended Pipeline
 
```python
# 1. Load data
import pandas as pd
df = pd.read_csv("fact_po_descriptions.csv")
# Keep only clean rows
df_clean = df[(df["word_count"] >= 20) & (df["char_count"] < 2000)].copy()
 
# 2. Embed with sentence-transformers (faster than raw BERT for clustering)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")   # or "all-mpnet-base-v2" for quality
embeddings = model.encode(df_clean["description_text"].tolist(),
                           show_progress_bar=True, batch_size=64)
# shape: (N, 384)
 
# 3. Alternatively: raw BERT pooled embeddings
from transformers import BertTokenizer, BertModel
import torch
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
 
def bert_embed(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=256, return_tensors="pt")
        with torch.no_grad():
            out = bert(**enc)
        cls_embs = out.last_hidden_state[:, 0, :]   # [CLS] token
        all_embs.append(cls_embs.numpy())
    return np.vstack(all_embs)
 
embeddings = bert_embed(df_clean["description_text"].tolist())
 
# 4. Dimensionality reduction
import umap
reducer = umap.UMAP(n_components=2, random_state=42)
emb_2d  = reducer.fit_transform(embeddings)
 
# 5. Clustering
from sklearn.cluster import KMeans
km = KMeans(n_clusters=8, random_state=42, n_init=20)
df_clean["pred_cluster"] = km.fit_predict(embeddings)
 
# 6. Evaluate against ground truth
from sklearn.metrics import adjusted_rand_score, silhouette_score
ari = adjusted_rand_score(df_clean["true_cluster_id"], df_clean["pred_cluster"])
sil = silhouette_score(embeddings, df_clean["pred_cluster"], sample_size=500)
print(f"ARI: {ari:.3f}  |  Silhouette: {sil:.3f}")
 
# 7. Visualise
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                     c=df_clean["true_cluster_id"], cmap="tab10", s=8, alpha=0.7)
plt.colorbar(scatter, label="Cluster")
plt.title("BERT Embeddings — UMAP Projection of PO Descriptions")
plt.tight_layout(); plt.show()
```
 
### Expected Results
- Silhouette score (sentence-transformers): ~0.45–0.62
- ARI vs ground truth: ~0.55–0.75 (clusters intentionally overlap at boundaries)
- UMAP plot: 8 visible islands, with partial merging of clusters 1/4 (both crisis-driven)
  and clusters 2/7 (both operational/transactional vocabulary)
 
### Data Quality Tasks for NLP
- `word_count == 0` → ~0.5% missing texts (handle before embedding)
- `char_count < 50` → truncated rows (filter or flag)
- Near-duplicate texts (~1%) → detect with cosine similarity before clustering
'''
 
with open(f"{DATA_DIR}/BERT_USAGE_GUIDE.md", "w") as f:
    f.write(GUIDE)
print(f"\n✓ BERT_USAGE_GUIDE.md saved.")